# 🚢 Titanic Disaster: Survival Prediction & End-to-End ML Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Domain:** Data Analysis & Machine Learning | **Competition:** Kaggle Titanic

---

## 📌 Project Overview
This project demonstrates an end-to-end Data Science pipeline on the Titanic dataset:
1. **Exploratory Data Analysis (EDA):** Uncover survival disparities across gender, socioeconomic class, and age.
2. **Feature Engineering & Smart Imputation:** Extract honorific titles to accurately impute missing ages without demographic distortion.
3. **Model Benchmarking:** Compare a heuristic baseline against Logistic Regression and Random Forest ensembles using Stratified 5-Fold Cross-Validation.
4. **Kaggle Submission:** Generate final predictions for unseen test instances.

## 1. Setup & Environment
Import essential libraries for numerical computation, visualization, and modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries successfully loaded!')

## 2. Load Datasets
We support both local files and direct downloading via GitHub raw URLs (convenient for Google Colab).

In [ ]:
try:
    train_df = pd.read_csv('data/train.csv')
    test_df = pd.read_csv('data/test.csv')
except Exception:
    train_url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    test_url = 'https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/test.csv'
    train_df = pd.read_csv(train_url)
    test_df = pd.read_csv(test_url)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head(3)

## 3. Exploratory Data Analysis (EDA)
Examine how gender and passenger class interacted during the evacuation.

In [ ]:
# Overall Survival Base Rate
overall_survival = train_df['Survived'].mean() * 100
print(f'Overall Historical Survival Rate: {overall_survival:.2f}%')

# Cross-tabulation: Survival % by Pclass and Sex
pivot_table = train_df.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean') * 100
print('\nSurvival Rate (%) by Class and Sex:')
print(pivot_table.round(2))

# Heuristic Baseline Accuracy (Women live, Men perish)
baseline_pred = (train_df['Sex'] == 'female').astype(int)
baseline_acc = (baseline_pred == train_df['Survived']).mean() * 100
print(f'\nBaseline Accuracy: {baseline_acc:.2f}% (Our ML models must beat this threshold)')

## 4. Feature Engineering & Preprocessing
- Extract Honorific Titles (`Mr`, `Mrs`, `Miss`, `Master`) from passenger names.
- Conditionally impute missing ages by Title median (e.g. Master median is 3.5 yrs, Mr is 30.0 yrs).
- Create `FamilySize` and `IsAlone` indicators.
- Apply log transformation on `Fare` to normalize heavy right skewness.

In [ ]:
def engineer_features(df, is_train=True):
    data = df.copy()
    
    # 1. Title Extraction
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    rare = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
    data['Title'] = data['Title'].replace(rare, 'Rare')
    data['Title'] = data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    
    # 2. Smart Age Imputation
    title_age_median = {'Master': 3.5, 'Miss': 21.0, 'Mr': 30.0, 'Mrs': 35.0, 'Rare': 48.5}
    for title, age in title_age_median.items():
        data.loc[(data['Age'].isnull()) & (data['Title'] == title), 'Age'] = age
    data['Age'] = data['Age'].fillna(data['Age'].median())
    
    # 3. Categorical and Fare Imputation
    data['Embarked'] = data['Embarked'].fillna('S')
    data['Fare'] = data['Fare'].fillna(data[data['Pclass'] == 3]['Fare'].median())
    
    # 4. Family Interaction Features
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    
    # 5. Skewness Reduction
    data['LogFare'] = np.log1p(data['Fare'])
    
    # 6. Categorical Encoding
    data['Sex_Code'] = data['Sex'].map({'male': 0, 'female': 1}).astype(int)
    data = pd.get_dummies(data, columns=['Embarked', 'Title'], drop_first=True, dtype=int)
    
    feature_cols = [
        'Pclass', 'Sex_Code', 'Age', 'SibSp', 'Parch',
        'LogFare', 'FamilySize', 'IsAlone'
    ] + [c for c in data.columns if c.startswith(('Embarked_', 'Title_'))]
    
    if is_train:
        return data[feature_cols], data['Survived']
    else:
        return data[feature_cols], data['PassengerId']

X_train, y_train = engineer_features(train_df, is_train=True)
X_test, test_ids = engineer_features(test_df, is_train=False)

# Align test columns with training features
for col in set(X_train.columns) - set(X_test.columns):
    X_test[col] = 0
X_test = X_test[X_train.columns]

print(f'Feature engineering complete: {X_train.shape[1]} features ready!')

## 5. Model Training & 5-Fold Stratified Cross-Validation
Evaluate Logistic Regression (Linear baseline) and Random Forest Classifier (Tree ensemble).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Logistic Regression
lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
lr_scores = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring='accuracy')

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_split=4, random_state=42)
rf_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='accuracy')

print('=== 5-Fold Cross-Validation Accuracy ===')
print(f'Baseline (Heuristic):     {baseline_acc:.2f}%')
print(f'Logistic Regression:      {lr_scores.mean() * 100:.2f}% (+/- {lr_scores.std() * 100:.2f}%)')
print(f'Random Forest (Depth=5):  {rf_scores.mean() * 100:.2f}% (+/- {rf_scores.std() * 100:.2f}%)')

## 6. Model Interpretation: Feature Importance
Inspect the factors that contributed most heavily to the Random Forest model decisions.

In [ ]:
rf_model.fit(X_train, y_train)
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette='viridis', legend=False)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

## 7. Kaggle Submission Generation
Generate the final `submission.csv` conforming to the competition requirements.

In [ ]:
test_predictions = rf_model.predict(X_test)
submission = pd.DataFrame({'PassengerId': test_ids, 'Survived': test_predictions})
submission.to_csv('submission.csv', index=False)

print(f'Submission file saved with {len(submission)} rows!')
print(f'Predicted Survival Ratio: {(submission["Survived"] == 1).mean() * 100:.2f}%')
submission.head()